In [1]:
import tensorflow as tf

# Load the MNIST dataset
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Combine the training and test data into a single data variable
data = (x_train, y_train), (x_test, y_test)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 10s 1us/step


In [4]:
import tensorflow as tf
import numpy as np

def scale_image(image, scale):
    return tf.image.resize(image, [int(image.shape[0] * scale), int(image.shape[1] * scale)])

def rotate_image(image, angle):
    return tf.image.rot90(image, k=angle)

def add_noise(image, noise_factor=0.5):
    noise = tf.random.normal(shape=tf.shape(image), mean=0.0, stddev=noise_factor, dtype=tf.float32)
    return tf.clip_by_value(image + noise, 0.0, 1.0)

def augment_image(image):
    scale = np.random.uniform(0.8, 1.2)
    angle = np.random.randint(0, 4)
    noise_factor = np.random.uniform(0.1, 0.5)
    
    image = tf.expand_dims(image, axis=-1)  # Add channel dimension
    image = scale_image(image, scale)
    image = rotate_image(image, angle)
    image = add_noise(image, noise_factor)
    
    return image

original_shape = x_train.shape[1:3]

aug_train_x = np.array([tf.image.resize(augment_image(tf.convert_to_tensor(img, dtype=tf.float32)), original_shape).numpy() for img in x_train])
aug_test_x = np.array([tf.image.resize(augment_image(tf.convert_to_tensor(img, dtype=tf.float32)), original_shape).numpy() for img in x_test])

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# Define the model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

# Print the model summary
model.summary()

c:\Users\singh\.conda\envs\RL\lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 121,930 (476.29 KB)

 Trainable params: 121,930 (476.29 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
import pickle

# Convert labels to categorical one-hot encoding
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

# Compile the model
model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(aug_train_x, y_train_cat, epochs=10, validation_data=(aug_test_x, y_test_cat))

# Save the model to a pickle file
with open('trained_model.pkl', 'wb') as file:
    pickle.dump(model, file)

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 19s 9ms/step - accuracy: 0.6709 - loss: 0.9876 - val_accuracy: 0.8934 - val_loss: 0.3408
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - accuracy: 0.9002 - loss: 0.3150 - val_accuracy: 0.9178 - val_loss: 0.2549
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - accuracy: 0.9335 - loss: 0.2094 - val_accuracy: 0.9346 - val_loss: 0.2083
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - accuracy: 0.9496 - loss: 0.1601 - val_accuracy: 0.9369 - val_loss: 0.2001
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - accuracy: 0.9583 - loss: 0.1312 - val_accuracy: 0.9451 - val_loss: 0.1848
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.9639 - loss: 0.1092 - val_accuracy: 0.9400 - val_loss: 0.2011
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.9698 - loss: 0.0882 - val_accuracy: 0.9469 - val_loss: 0.1882
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - accuracy: 0.9748 - loss: 0

In [8]:
import pickle

# Load the model from the pickle file
with open('trained_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

# Evaluate the model on the test data
loss, accuracy = loaded_model.evaluate(aug_test_x, y_test_cat)
print(f'Accuracy: {accuracy * 100:.2f}%')

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9342 - loss: 0.2521
Accuracy: 94.18%
